This notebook was started using the Advanced RAG cookbook from Huggingface. https://huggingface.co/learn/cookbook/advanced_rag

As we specialize our model for our research we will diverge from this proof of concept system

In [ ]:
%pip install -q torch langchain-text-splitters langchain-huggingface transformers accelerate bitsandbytes langchain sentence-transformers faiss-cpu openpyxl pacmap datasets langchain-community ragatouille jedi matplotlib plotly "nbformat>=4.2.0" ipywidgets

In [ ]:
from tqdm.notebook import tqdm
import pandas as pd
from typing import Optional, List, Tuple
import datasets
import matplotlib.pyplot as plt
import torch


In [ ]:
DATASET_NAME = "m-ric/huggingface_doc"
TOKENIZER_NAME = "thenlper/gte-small"

In [ ]:

ds = datasets.load_dataset(DATASET_NAME, split="train")
df = ds.to_pandas()
df.head()

In [ ]:
from langchain_core.documents import Document as LangchainDocument

RAW_KNOWLEDGE_BASE = [
    LangchainDocument(page_content=doc["text"], metadata={"source": doc["source"]})
    for doc in tqdm(ds)
]
print(RAW_KNOWLEDGE_BASE[0])

In [ ]:
from help_function import split_knowledge_base

docs_processed = split_knowledge_base(
    512,
    RAW_KNOWLEDGE_BASE,
    tokenizer_name=TOKENIZER_NAME,
)
print("Docs done")

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
lengths = [len(tokenizer.encode(doc.page_content)) for doc in tqdm(docs_processed)]
fig = pd.Series(lengths).hist()
plt.title("Distribution of document lengths in the knowledge base (in count of tokens)")
plt.show()

In [ ]:
from help_function import load_knowledge_base

KNOWLEDGE_VECTOR_DATABASE, embedding_model = load_knowledge_base(docs_processed=docs_processed)

In [ ]:
# Test the embedding model by embedding a sample query
user_query = "How to create a pipeline object?"
query_vector = embedding_model.embed_query(user_query)
query_vector

In [ ]:
# Add additional JSON data to the existing knowledge base
from help_function import add_json_to_knowledge_base

KNOWLEDGE_VECTOR_DATABASE = add_json_to_knowledge_base(
    path="test_data/test_data.json",
    knowledge_base=KNOWLEDGE_VECTOR_DATABASE,
)

In [ ]:
print("***** RETRIEVAL *****")

retrieved_docs = KNOWLEDGE_VECTOR_DATABASE.similarity_search(query=user_query, k=5)

print("***** RESULTS *****")
for idx, doc in enumerate(retrieved_docs):
    print("\n**** SOURCE ****")
    print(f"Source: {doc.metadata['source']}")
    print("\n**** CONTENT ****")
    print(f"Extract: {doc.page_content[:100]}...")

In [ ]:
from model_function import load_reader_model

READER_LLM = load_reader_model()
tokenizer = READER_LLM.tokenizer

#test reader
READER_LLM("Whats the Capitol of Canada?")

In [ ]:
prompt_in_chat_format = [
    {
        "role": "system",
        "content": """Using the information contained in the context,
give a comprehensive answer to the question.
Respond only to the question asked, response should be concise and relevant to the question.
Provide the number of the source document when relevant.
If the answer cannot be deduced from the context, do not give an answer.""",
    },
    {
        "role": "user",
        "content": """Context:
{context}
---
Now here is the question you need to answer.

Question: {question}""",
    },
]
RAG_PROMPT_TEMPLATE = tokenizer.apply_chat_template(
    prompt_in_chat_format, tokenize=False, add_generation_prompt=True
)
print(RAG_PROMPT_TEMPLATE)

In [ ]:
# TODO: Code subject to edit based on experimental needs

retrieved_docs_text = [
    doc.page_content for doc in retrieved_docs
]  # We only need the text of the documents
context = "\nExtracted documents:\n"
context += "".join(
    [f"Document {str(i)}:::\n" + doc for i, doc in enumerate(retrieved_docs_text)]
)

final_prompt = RAG_PROMPT_TEMPLATE.format(
    question="How to create a pipeline object?", context=context
)

# Redact an answer
answer = READER_LLM(final_prompt)[0]["generated_text"]
print(answer)

In [ ]:
# Previous code cells are for RAG pipeline. The following code cells are for visualization of the knowledge base and the query in 2D space using PaCMAP.
# Currently not needed for the RAG pipeline, but can be useful for understanding the distribution of the knowledge base and the query in the embedding space.
import pacmap
import numpy as np
import plotly.express as px

embedding_projector = pacmap.PaCMAP(
    n_components=2, n_neighbors=None, MN_ratio=0.5, FP_ratio=2.0, random_state=1
)

embeddings_2d = [
    list(KNOWLEDGE_VECTOR_DATABASE.index.reconstruct_n(idx, 1)[0])
    for idx in range(len(docs_processed))
] + [query_vector]

# Fit the data (the index of transformed data corresponds to the index of the original data)
documents_projected = embedding_projector.fit_transform(
    np.array(embeddings_2d), init="pca"
)

In [ ]:
df = pd.DataFrame.from_dict(
    [
        {
            "x": documents_projected[i, 0],
            "y": documents_projected[i, 1],
            "source": docs_processed[i].metadata["source"].split("/")[1],
            "extract": docs_processed[i].page_content[:100] + "...",
            "symbol": "circle",
            "size_col": 4,
        }
        for i in range(len(docs_processed))
    ]
    + [
        {
            "x": documents_projected[-1, 0],
            "y": documents_projected[-1, 1],
            "source": "User query",
            "extract": user_query,
            "size_col": 100,
            "symbol": "star",
        }
    ]
)

# Visualize the embedding
fig = px.scatter(
    df,
    x="x",
    y="y",
    color="source",
    hover_data="extract",
    size="size_col",
    symbol="symbol",
    color_discrete_map={"User query": "black"},
    width=1000,
    height=700,
)
fig.update_traces(
    marker=dict(opacity=1, line=dict(width=0, color="DarkSlateGrey")),
    selector=dict(mode="markers"),
)
fig.update_layout(
    legend_title_text="<b>Chunk source</b>",
    title="<b>2D Projection of Chunk Embeddings via PaCMAP</b>",
)
fig.show()